In [1]:
# import packages
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from tqdm import tqdm

In [2]:
# set device
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(f"Training with devive = {device}.")

Training with devive = mps.


In [3]:
# get the dataset

# set path variable
data = "./data"

# define transforms
train_transform = v2.Compose([
    v2.RandomResizedCrop(size = (224, 224), antialias = True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale = True),
    v2.Normalize(mean = [0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225])
])

val_transform = v2.Compose([
    v2.Resize(size = (224, 224), antialias = True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale = True),
    v2.Normalize(mean = [0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225])
])

# get data
train_dataset = torchvision.datasets.VOCSegmentation(
    root = data,
    year = "2012",
    download = True,
    transforms = train_transform
)

val_dataset = torchvision.datasets.VOCSegmentation(
    root = data,
    year = "2012",
    download = True,
    transforms = val_transform
)

classes = [
    "background",
    "aeroplane",
    "bicycle",
    "bird",
    "boat",
    "bottle",
    "bus",
    "car",
    "cat",
    "chair",
    "cow",
    "diningtable",
    "dog",
    "horse",
    "motorbike",
    "person",
    "potted plant",
    "sheep",
    "sofa",
    "train",
    "tv/monitor"
]


# get loaders
train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 128, shuffle = False)

Using downloaded and verified file: ./data/VOCtrainval_11-May-2012.tar
Extracting ./data/VOCtrainval_11-May-2012.tar to ./data
Using downloaded and verified file: ./data/VOCtrainval_11-May-2012.tar
Extracting ./data/VOCtrainval_11-May-2012.tar to ./data


In [4]:
# load model, optimizer, and loss function

# model
from models.ResidualUNet import ResidualUNet
model = ResidualUNet(num_classes = 21)

# optimizer
optimizer = optim.Adam(params = model.parameters(),lr = 0.001, weight_decay = 1e-4)

# loss function
loss_func = nn.CrossEntropyLoss(ignore_index = 255)

In [5]:
# train model
from utilities.train import train_step, evaluate
from torchmetrics.classification import MulticlassJaccardIndex


# train
num_epochs = 1
best_model = None
best_iou = 0

for epoch in range(num_epochs) :

    # metric to optimize 
    train_metric = MulticlassJaccardIndex(num_classes=21, ignore_index=255).to(device)
    val_metric = MulticlassJaccardIndex(num_classes=21, ignore_index=255).to(device)

    # instantiate pbar
    pbar = tqdm(train_loader, desc = f"Epoch {epoch + 1}/{num_epochs}", leave = True)

    # train 1 epoch
    model, avg_epoch_loss, epoch_iou = train_step(model, loss_func, optimizer,
               device, pbar, train_metric) 

    # evaluate on validation set
    eval_metric = evaluate(model, val_loader, device, val_metric) 

Epoch 1/1:   0%|          | 0/12 [00:01<?, ?it/s]


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of dimension: 4